In [2]:
# import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Library for profiling
from ydata_profiling import ProfileReport

# visual configuration
sns.set_theme(style="whitegrid")
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

c:\dev\Portfolio\saas-churn-analytics\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully.


C:\Users\houss\AppData\Local\Temp\ipykernel_9024\40661553.py:9: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [3]:
# Data paths
TRAIN_PATH = "../data/raw/train.csv"
TEST_PATH = "../data/raw/test.csv"

# Load the data
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

print(f"Train set shape: {df_train.shape}")
print(f"Test set shape: {df_test.shape}")
df_train.head()

Train set shape: (243787, 21)
Test set shape: (104480, 20)


,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID,Churn
0,20,11.055215,221.104302,Premium,Mailed check,No,Both,No,Mobile,36.758104,63.531377,10,Sci-Fi,2.176498,4,Male,3,No,No,CB6SXPNVZA,0
1,57,5.175208,294.986882,Basic,Credit card,Yes,Movies,No,Tablet,32.450568,25.725595,18,Action,3.478632,8,Male,23,No,Yes,S7R2G87O09,0
2,73,12.106657,883.785952,Basic,Mailed check,Yes,Movies,No,Computer,7.395160,57.364061,23,Fantasy,4.238824,6,Male,1,Yes,Yes,EASDC20BDT,0
3,32,7.263743,232.439774,Basic,Electronic check,No,TV Shows,No,Tablet,27.960389,131.537507,30,Drama,4.276013,2,Male,24,Yes,Yes,NPF69NT69N,0
4,57,16.953078,966.325422,Premium,Electronic check,Yes,TV Shows,No,TV,20.083397,45.356653,20,Comedy,3.616170,4,Female,0,No,No,4LGYPK7VOL,0


In [4]:

# 1. verify missing values
missing_values = df_train.isnull().sum()
missing_percent = (missing_values / len(df_train)) * 100
missing_df = pd.DataFrame({'Missing Values': missing_values, 'Percentage (%)': missing_percent})
print("--- Missing Values ---")
print(missing_df[missing_df['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False))

# 2. verify duplicates
duplicates = df_train.duplicated().sum()
print(f"\n--- Duplicates ---")
print(f"Number of duplicate rows: {duplicates}")

# 3. verify data types
print("\n--- Data Types Requiring Attention  ---")
print(df_train.dtypes.value_counts())

--- Missing Values ---
Empty DataFrame
Columns: [Missing Values, Percentage (%)]
Index: []

--- Duplicates ---
Number of duplicate rows: 0

--- Data Types Requiring Attention  ---
object     11
int64       5
float64     5
Name: count, dtype: int64


In [5]:
# cell for generating the data profiling report
# Attention : take 1 to 2 minutes to generate the report depending on the size of the dataset

print("Generating Data Profiling Report...")
profile = ProfileReport(df_train, title="SaaS Churn Data Profiling Report", explorative=True)

# Save the report to an HTML file
profile.to_file("../reports/figures/data_profile_train.html")
print("Report generated and saved under: '../reports/figures/data_profile_train.html'")

# Optionnel : for displaying the report in a Jupyter Notebook (uncomment the line below if using a notebook)
# profile.to_widgets()

Generating Data Profiling Report...


Summarize dataset:  61%|██████    | 66/109 [00:43<00:28,  1.53it/s, scatter UserRating, AverageViewingDuration]              


KeyboardInterrupt: 

In [6]:
TARGET = 'Churn' 

if TARGET in df_train.columns:
    # 1. Overall churn rate
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df_train, x=TARGET, palette='viridis')
    plt.title('Target variable distribution')

    plt.savefig('../reports/figures/01_target_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()


    # 2. support tickets impact on churn
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df_train, x=TARGET, y='SupportTicketsPerMonth', palette='Set2')
    plt.title('support tickets impact on churn')

    plt.savefig('../reports/figures/02_support_tickets_impact_churn.png', dpi=300, bbox_inches='tight')
    plt.show()


    # 3. Viewing hours impact on churn
    plt.figure(figsize=(8, 5))
    sns.kdeplot(data=df_train, x='ViewingHoursPerWeek', hue=TARGET, fill=True, common_norm=False, palette='crest')
    plt.title('Distribution of Viewing Hours by Churn Status')

    plt.savefig('../reports/figures/03_viewing_hours_impact_churn.png', dpi=300, bbox_inches='tight')
    plt.show()

else:
    print(f" '{TARGET}' not found. Please verify its exact name (e.g., 'Churn_Yes').")

In [8]:
def detect_outliers_iqr(df, column):
    q1, q3 = df[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[column] < lower) | (df[column] > upper)).sum()
    return {
        "column": column, "lower_bound": lower, "upper_bound": upper,
        "n_outliers": n_outliers, "pct_outliers": round(n_outliers / len(df) * 100, 2)
    }

numeric_cols_to_check = [
    "AccountAge", "MonthlyCharges", "TotalCharges", "ViewingHoursPerWeek",
    "AverageViewingDuration", "ContentDownloadsPerMonth", "UserRating",
    "SupportTicketsPerMonth", "WatchlistSize"
]

outliers_summary = pd.DataFrame([detect_outliers_iqr(df_train, col) for col in numeric_cols_to_check])
outliers_summary = outliers_summary.sort_values("pct_outliers", ascending=False)
outliers_summary

,column,lower_bound,upper_bound,n_outliers,pct_outliers
2,TotalCharges,-811.108476,2229.572864,741,0.3
0,AccountAge,-60.000000,180.000000,0,0.0
1,MonthlyCharges,-2.510882,27.487586,0,0.0
3,ViewingHoursPerWeek,-18.419212,59.402561,0,0.0
4,AverageViewingDuration,-82.906085,267.196528,0,0.0
5,ContentDownloadsPerMonth,-25.500000,74.500000,0,0.0
6,UserRating,-1.001103,7.004114,0,0.0
7,SupportTicketsPerMonth,-5.500000,14.500000,0,0.0
8,WatchlistSize,-12.000000,36.000000,0,0.0


## Summary & Key Findings

- **Shape**: 243,787 rows / 21 columns. No missing values, no duplicate rows —
  the dataset is structurally clean.
- **Domain context**: subscription-based streaming/media service (not B2B SaaS) —
  churn drivers are expected to center on content engagement (viewing hours,
  downloads, watchlist) rather than B2B feature adoption.
- **Target balance**: churn rate = **18.12%** (No: 81.87%, Yes: 18.12%).
  Confirmed class imbalance (~1 churner per 3.4 retained customers) →
  will require `class_weight="balanced"` and/or SMOTE, and evaluation must
  rely on Recall/F1/ROC-AUC rather than accuracy.
- **Correlated features**: `AccountAge` and `TotalCharges` are strongly
  correlated (expected, cumulative billing) → will assess multicollinearity
  before using linear models, and consider engineering a ratio feature
  (e.g. `TotalCharges / AccountAge`) instead of keeping both raw.
- **Zero-heavy columns**: `SupportTicketsPerMonth` (10.0%), `WatchlistSize`
  (4.0%), `ContentDownloadsPerMonth` (2.0%) — genuine "inactive on this
  dimension" values, not missing data; kept as-is.
- **Strongest early engagement signal**: `ViewingHoursPerWeek` — churned
  customers concentrate at low weekly usage, retained customers show a
  flatter, higher-usage distribution. Strong candidate for top feature
  importance later.
- **Moderate signal**: `SupportTicketsPerMonth` — churners skew slightly
  higher but with heavy overlap; likely more useful combined with other
  features than alone.

**Next step:** proceed to `02_eda.ipynb` (deeper bivariate analysis across
all categorical features) and `src/data/clean_data.py` (cleaning strategy:
outlier handling on charges/viewing hours, encoding of categorical features,
correlation-driven feature selection).

## Outlier Detection (IQR Method)

| Column | Lower Bound | Upper Bound | Outliers | % of Data |
|---|---|---|---|---|
| TotalCharges | -811.11 | 2229.57 | 741 | 0.30% |
| AccountAge | -60.00 | 180.00 | 0 | 0.00% |
| MonthlyCharges | -2.51 | 27.49 | 0 | 0.00% |
| ViewingHoursPerWeek | -18.42 | 59.40 | 0 | 0.00% |
| AverageViewingDuration | -82.91 | 267.20 | 0 | 0.00% |
| ContentDownloadsPerMonth | -25.50 | 74.50 | 0 | 0.00% |
| UserRating | -1.00 | 7.00 | 0 | 0.00% |
| SupportTicketsPerMonth | -5.50 | 14.50 | 0 | 0.00% |
| WatchlistSize | -12.00 | 36.00 | 0 | 0.00% |

**Interpretation:**

- Almost no outliers detected across the dataset (only `TotalCharges`,
  with 0.3% of rows / 741 customers flagged). This is a very low rate —
  well below the ~1-3% typically seen in real-world billing data.
- The near-total absence of outliers, combined with the earlier finding of
  0% missing values and 0% duplicates, suggests this dataset may be
  **synthetically generated or heavily pre-cleaned** rather than raw
  production data. This doesn't invalidate the project, but it means the
  "outlier handling" step in cleaning will be minimal, and results should
  be framed as a portfolio/methodology demonstration rather than a claim
  about real customer behavior.
- Several IQR bounds are not meaningful in business terms (e.g. `AccountAge`
  lower bound of -60 months, `UserRating` upper bound of 7 on a 1-5 scale).
  This happens when a variable's distribution is close to uniform/bounded —
  the IQR range becomes wide relative to the actual data range, so the
  method correctly finds nothing extreme. It confirms these variables
  don't need outlier treatment, rather than signaling a flaw in the method.
- **Decision for the cleaning step:** the 741 `TotalCharges` outliers will
  be inspected individually (not blindly capped) — likely legitimate
  long-tenure/high-value customers rather than data errors, consistent
  with `TotalCharges` being correlated with `AccountAge`. No winsorizing
  planned elsewhere given the 0% outlier rate on all other columns.




